# Підготовка даних

```
текст + спани  →  токени  →  теги O/B/I  →  id токенів  →  батчі з padding
```

In [1]:
import os, re, ast, pickle, string
from collections import Counter
from typing import List
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

SEED     = 42
N_BIG    = 200_000
MAX_LEN  = 256
MIN_FREQ = 2
BPE_SIZE = 16_000

PAD, UNK = 0, 1
LABEL2ID = {'O': 0, 'B': 1, 'I': 2}
IGNORE   = -100

## 1. Завантаження даних і поділу train / valid

In [2]:
files = [os.path.join(d, f) for d, _, fs in os.walk('/kaggle/input') for f in fs]

def find(word, exts=('.parquet', '.csv')):
    return [f for f in files if word in os.path.basename(f).lower() and f.endswith(exts)][0]

def load(path, n=None):
    if path.endswith('.parquet'):
        if n is None:
            return pd.read_parquet(path)
        return next(pq.ParquetFile(path).iter_batches(batch_size=n)).to_pandas()
    return pd.read_csv(path, nrows=n)

def to_list(x):
    if isinstance(x, str):
        x = ast.literal_eval(x)
    return [tuple(s) for s in x]

uk   = load(find('uk_geo_dataset.csv'), N_BIG)
ru   = load(find('ru_geo'), N_BIG)
lab  = load(find('labeling'))
test = load(find('test'))

for df in (uk, ru, lab):
    df['loc_markers'] = df['loc_markers'].apply(to_list)

uk['source'], ru['source'] = 'uk', 'ru'
uk['row'], ru['row'] = uk.index, ru.index
big = pd.concat([uk[['text', 'loc_markers', 'source', 'row']],
                 ru[['text', 'loc_markers', 'source', 'row']]], ignore_index=True)

splits = pd.read_csv(find('splits.csv'))
big = big.merge(splits, on=['source', 'row'])
big = big[big['split'] == 'train'].reset_index(drop=True)

val = pd.read_csv(find('valid_docs.csv'))
val['loc_markers'] = val['loc_markers'].apply(to_list)
val['locs'] = val.apply(lambda r: [r['text'][a:b] for a, b in r['loc_markers']], axis=1)

lab['locs'] = lab.apply(lambda r: [r['text'][a:b] for a, b in r['loc_markers']], axis=1)
big['locs'] = big.apply(lambda r: [r['text'][a:b] for a, b in r['loc_markers']], axis=1)
print('train:', len(big), '| valid:', len(val), '| labeling:', len(lab), '| test:', len(test))

train: 373790 | valid: 2000 | labeling: 26 | test: 477


## 2. Токенізація

In [3]:
def tokenize_words(text):
    tokens, offsets = [], []
    for m in re.finditer(r"\w+|[^\w\s]", text):
        tokens.append(m.group())
        offsets.append((m.start(), m.end()))
    return tokens, offsets

tokens, offsets = tokenize_words('Я живу в м. Київ, вул. Хрещатик 5')
print(tokens)
print(offsets)

['Я', 'живу', 'в', 'м', '.', 'Київ', ',', 'вул', '.', 'Хрещатик', '5']
[(0, 1), (2, 6), (7, 8), (9, 10), (10, 11), (12, 16), (16, 17), (18, 21), (21, 22), (23, 31), (32, 33)]


## 3. Теги: з спанів у O / B / I

In [4]:
def make_labels(offsets, spans):
    labels = ['O'] * len(offsets)
    for s_start, s_end in spans:
        first = True
        for i, (t_start, t_end) in enumerate(offsets):
            if t_start < s_end and t_end > s_start:
                labels[i] = 'B' if first else 'I'
                first = False
    return labels

text  = 'Я живу в м. Київ, вул. Хрещатик 5'
spans = [(12, 16), (23, 31)]
tokens, offsets = tokenize_words(text)
for t, l in zip(tokens, make_labels(offsets, spans)):
    print(f'{t:12s} {l}')

Я            O
живу         O
в            O
м            O
.            O
Київ         B
,            O
вул          O
.            O
Хрещатик     B
5            O


## 4. Зворотний шлях: з тегів у рядки локацій

In [5]:
def labels_to_strings(text, offsets, labels):
    result, start, end = [], None, None
    for (t_start, t_end), lab_ in zip(offsets, labels):
        if lab_ == 'B':
            if start is not None:
                result.append(text[start:end])
            start, end = t_start, t_end
        elif lab_ == 'I' and start is not None:
            end = t_end
        else:
            if start is not None:
                result.append(text[start:end])
            start, end = None, None
    if start is not None:
        result.append(text[start:end])
    return result

print(labels_to_strings(text, offsets, make_labels(offsets, spans)))

['Київ', 'Хрещатик']


### Перевірка: чи не губимо ми інформацію?

In [6]:
def hard_processing(text):
    text = re.sub(r"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r'\d', '', text)
    text = re.sub(r'\b\w\b\s?', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def comp_metric(y_true: List[List[str]], y_pred: List[List[str]]):
    assert len(y_true) == len(y_pred)
    tp, fp, fn, p = 0.0, 0.0, 0.0, 0.0
    for t, pr in zip(y_true, y_pred):
        t = set([hard_processing(s) for s in t])
        pr = set([hard_processing(s) for s in pr])
        tp += len(t & pr); fp += len(pr - t); fn += len(t - pr); p += len(t)
    if tp + fp == 0:
        precision = 1.0 if p == 0 else 0.0
    else:
        precision = tp / (tp + fp)
    if tp + fn == 0:
        recall = 1.0 if p == 0 else 0.0
    else:
        recall = tp / (tp + fn)
    f1 = 0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return round(f1, 8)

def roundtrip_score(df, tokenizer):
    preds = []
    for text, spans in zip(df['text'], df['loc_markers']):
        _, offs = tokenizer(text)
        preds.append(labels_to_strings(text, offs, make_labels(offs, spans)))
    return comp_metric(df['locs'].tolist(), preds)

print('word-level, максимум метрики:', roundtrip_score(lab, tokenize_words))

word-level, максимум метрики: 1.0


## 5. Готуємо всі набори

In [7]:
def prepare(df, tokenizer, with_labels=True):
    out = {'tokens': [], 'offsets': []}
    if with_labels:
        out['labels'] = []
    for i, text in enumerate(df['text']):
        tokens, offsets = tokenizer(text)
        out['tokens'].append(tokens)
        out['offsets'].append(offsets)
        if with_labels:
            tags = make_labels(offsets, df['loc_markers'].iloc[i])
            out['labels'].append([LABEL2ID[t] for t in tags])
    return out

word_big  = prepare(big,  tokenize_words)
word_val  = prepare(val,  tokenize_words)
word_lab  = prepare(lab,  tokenize_words)
word_test = prepare(test, tokenize_words, with_labels=False)

lens = pd.Series([len(t) for t in word_test['tokens']])
print(f'токенів у тексті: mean={lens.mean():.1f}, max={lens.max()}, довших за {MAX_LEN}: {(lens > MAX_LEN).mean()*100:.1f}%')

токенів у тексті: mean=69.6, max=601, довших за 256: 2.7%


## 6. Словник і перетворення в числа (word-level)

In [8]:
counter = Counter(w.lower() for toks in word_big['tokens'] for w in toks)

word_vocab = {'<PAD>': PAD, '<UNK>': UNK}
for word, count in counter.items():
    if count >= MIN_FREQ:
        word_vocab[word] = len(word_vocab)
print('розмір словника:', len(word_vocab))

def words_to_ids(tokens):
    return [word_vocab.get(w.lower(), UNK) for w in tokens]

word_big['ids']  = [words_to_ids(t) for t in word_big['tokens']]
word_val['ids']  = [words_to_ids(t) for t in word_val['tokens']]
word_lab['ids']  = [words_to_ids(t) for t in word_lab['tokens']]
word_test['ids'] = [words_to_ids(t) for t in word_test['tokens']]

for name, d in [('valid', word_val), ('labeling', word_lab), ('test', word_test)]:
    print(f'частка <UNK> у {name}: {np.mean([i == UNK for ids in d["ids"] for i in ids])*100:.1f}%')

розмір словника: 179708
частка <UNK> у valid: 4.1%
частка <UNK> у labeling: 7.7%
частка <UNK> у test: 7.8%


## 7. BPE-токенізація (другий варіант)

In [9]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers

bpe = Tokenizer(models.BPE(unk_token='<UNK>'))
bpe.normalizer = normalizers.Lowercase()
bpe.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(vocab_size=BPE_SIZE, special_tokens=['<PAD>', '<UNK>'])

bpe.train_from_iterator(list(big['text']), trainer)
print('розмір словника BPE:', bpe.get_vocab_size())

def tokenize_bpe(text):
    enc = bpe.encode(text)
    return enc.tokens, enc.offsets

print(tokenize_bpe('Я живу в м. Київ, вул. Хрещатик 5')[0])
print('word-level, максимум метрики на valid:', roundtrip_score(val, tokenize_words))
print('BPE,        максимум метрики на valid:', roundtrip_score(val, tokenize_bpe))




розмір словника BPE: 16000
['я', 'живу', 'в', 'м', '.', 'київ', ',', 'вул', '.', 'хреща', 'тик', '5']
word-level, максимум метрики на valid: 0.99694773
BPE,        максимум метрики на valid: 0.9977108


In [10]:
bpe_big  = prepare(big,  tokenize_bpe)
bpe_val  = prepare(val,  tokenize_bpe)
bpe_lab  = prepare(lab,  tokenize_bpe)
bpe_test = prepare(test, tokenize_bpe, with_labels=False)

for d in (bpe_big, bpe_val, bpe_lab, bpe_test):
    d['ids'] = [[bpe.token_to_id(t) if bpe.token_to_id(t) is not None else UNK for t in toks] for toks in d['tokens']]

## 8. Dataset і батчі з padding

In [11]:
class NERDataset(Dataset):
    def __init__(self, ids, labels=None):
        self.ids, self.labels = ids, labels
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        item = {'ids': torch.tensor(self.ids[i][:MAX_LEN])}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[i][:MAX_LEN])
        return item

def collate(batch):
    out = {
        'input_ids': pad_sequence([b['ids'] for b in batch], batch_first=True, padding_value=PAD),
        'lengths':   torch.tensor([len(b['ids']) for b in batch]),
    }
    if 'labels' in batch[0]:
        out['labels'] = pad_sequence([b['labels'] for b in batch], batch_first=True, padding_value=IGNORE)
    return out

ds = NERDataset(word_lab['ids'], word_lab['labels'])
batch = next(iter(DataLoader(ds, batch_size=4, collate_fn=collate)))
for k, v in batch.items():
    print(k, tuple(v.shape))
print(batch['input_ids'][0][:15])
print(batch['labels'][0][:15])

input_ids (4, 127)
lengths (4,)
labels (4, 127)
tensor([   83, 30323,    14, 38307,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0])
tensor([   0,    0,    0,    0, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100])


## 9. Претреновані embeddings

In [12]:
VEC_PATH = None

def load_embeddings(vec_path, vocab, dim=300):
    matrix = np.random.normal(0, 0.1, (len(vocab), dim)).astype('float32')
    matrix[PAD] = 0
    found = 0
    with open(vec_path, encoding='utf8') as f:
        next(f)
        for line in f:
            parts = line.rstrip().split(' ')
            word = parts[0]
            if word in vocab and len(parts) == dim + 1:
                matrix[vocab[word]] = np.array(parts[1:], dtype='float32')
                found += 1
    print(f'знайдено векторів: {found} з {len(vocab)}')
    return matrix

emb_matrix = load_embeddings(VEC_PATH, word_vocab) if VEC_PATH else None

## 10. Збереження

In [13]:
def pack(d, extra=None):
    d = {k: v for k, v in d.items() if k != 'tokens'}
    if extra:
        d.update(extra)
    return d

lab_extra = {'locs': lab['locs'].tolist()}
val_extra = {'locs': val['locs'].tolist()}

prepared = {
    'word': {'vocab': word_vocab, 'big': pack(word_big), 'val': pack(word_val, val_extra), 'lab': pack(word_lab, lab_extra),
             'test': pack(word_test, {'text_id': test['text_id'].tolist()}), 'emb_matrix': emb_matrix},
    'bpe':  {'big': pack(bpe_big), 'val': pack(bpe_val, val_extra), 'lab': pack(bpe_lab, lab_extra),
             'test': pack(bpe_test, {'text_id': test['text_id'].tolist()}), 'vocab_size': bpe.get_vocab_size(),
             'tokenizer_json': bpe.to_str()},
    'val_text': val['text'].tolist(),
    'lab_text': lab['text'].tolist(),
    'test_text': test['text'].tolist(),
}

with open('/kaggle/working/prepared.pkl', 'wb') as f:
    pickle.dump(prepared, f)
bpe.save('/kaggle/working/bpe.json')
print('збережено')

збережено


## Підсумок

**Що зроблено.** Тексти й спани локацій перетворені на числа: токени → теги `O/B/I` → id → батчі з padding. Усе збережено в `prepared.pkl` (дані) та `bpe.json` (токенізатор).

**Дані.**

| Набір | Розмір | Роль |
|---|---|---|
| `big` | 373 790 | навчання |
| `val` | 2 000 | склеєний valid, порівняння експериментів |
| `lab` | 26 | контроль, у навчання не йде |
| `test` | 477 | передбачення для сабміту |

**Токенізація.**
- Word-level: слово або знак пунктуації = токен, регістр ігнорується. Словник 179 708 слів (частота ≥ 2).
- BPE: словник 16 000, навчений лише на train. Рідкісні слова ділить на частини (`Хрещатик` → `хреща` + `тик`), що допомагає з відмінковими формами.

**Невідомі слова (`<UNK>`) у word-level.**

| Набір | `<UNK>` |
|---|---|
| valid | 4,1 % |
| labeling | 7,7 % |
| test | 7,8 % |

На тесті невідомих слів майже вдвічі більше, ніж на valid: у тесті більше рідкісних назв, адрес і відмінкових форм.